## MyDriveをマウント

In [56]:
#####  MyDriveのマウント

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##  'MyDrive/receipts/処理結果/2026-02/csv'内のcsvファイルへのパスを取得

In [57]:
#####  方法1：ディレクトリ内のすべてのcsvファイル名を取得し、上位部分と結合する

import os

# パスの指定
path = '/content/drive/MyDrive/receipts/処理結果/2026-02/csv'

# ファイル名のリストを取得
file_names = os.listdir(path)

# 取得したファイル名を表示
for file_name in file_names:
    print(file_name)

print()

# フルパスに変換
csv_paths = [os.path.join(path, f) for f in os.listdir(path)]  # 内包表記を利用
print(csv_paths)

20260402.csv

['/content/drive/MyDrive/receipts/処理結果/2026-02/csv/20260402.csv']


## 試しにpandasで読み込んでみる

In [58]:
#####  試しにpandasで読み込んでみる

import pandas as pd

# 先ほどのglobなどで取得したリストの1番目（インデックス0）を読み込む
df = pd.read_csv(csv_paths[0])

# 内容の確認
print(f"読み込んだファイル: {csv_paths[0]}")
display(df.head())

読み込んだファイル: /content/drive/MyDrive/receipts/処理結果/2026-02/csv/20260402.csv


,日付,科目,摘要,支出金額,処理対象pdfファイル名
0,2026/02/05,福利厚生費,新潟県けんこう財団_健康診断代,3686,Image_20260317_0002.pdf
1,2026/02/05,旅費交通費,ホテル法華クラブ新潟長岡_宿泊費,10300,Image_20260317_0002.pdf
2,2026/02/01,接待交際費,酒屋いって_飲食代,22400,Image_20260317_0002.pdf
3,2026/03/20,接待交際費,味処なべ_飲食代,12000,Image_20260317_0002.pdf
4,2026/02/20,旅費交通費,つばめタクシー_タクシー代,1400,Image_20260317_0002.pdf


## 複数のcsvを読み込んで結合する

In [59]:
##### 複数のcsvを読み込んで結合する

# 各ファイルを読み込んでリストに格納
df_list = []
for path in csv_paths:
    temp_df = pd.read_csv(path)
    df_list.append(temp_df)

# すべてのDataFrameを縦方向に結合
df_concat = pd.concat(df_list, ignore_index=True)

# 結果の確認
print(f"結合したファイル数: {len(df_list)}")
print()
print(f"結合した領収書数: {len(df_concat)}")
print(len(df_concat))
print()

print('最初の3つ')
display(df_concat.head(3))
print()
print('最後の3つ')
display(df_concat.tail(3))

結合したファイル数: 1

結合した領収書数: 23
23

最初の3つ


,日付,科目,摘要,支出金額,処理対象pdfファイル名
0,2026/02/05,福利厚生費,新潟県けんこう財団_健康診断代,3686,Image_20260317_0002.pdf
1,2026/02/05,旅費交通費,ホテル法華クラブ新潟長岡_宿泊費,10300,Image_20260317_0002.pdf
2,2026/02/01,接待交際費,酒屋いって_飲食代,22400,Image_20260317_0002.pdf



最後の3つ


,日付,科目,摘要,支出金額,処理対象pdfファイル名
20,2026/02/12,旅費交通費,つばめタクシー_タクシー代,900,Image_20260317_0001.pdf
21,2026/02/14,旅費交通費,長岡タクシー_タクシー代,1200,Image_20260317_0001.pdf
22,2026/02/26,福利厚生費,新潟県けんこう財団_健康診断代,8800,Image_20260317_0001.pdf


## 結合したdf_concatの要素を適切な型に変換する

- 日付→をdatetime型
- 支出金額→整数型

In [60]:
# # 最初の行（0番目）を抽出し、各要素の型を確認
# print(df_concat.iloc[0].apply(type))

In [61]:
# 1. 日付列を datetime型に変換
df_concat['日付'] = pd.to_datetime(df_concat['日付'])

# 2. 金額・税額列から数値以外（カンマ等）を除去して整数型(int)に変換
# ※数値として読み取れない値（欠損値など）がある場合は .fillna(0) などで補完が必要
df_concat['支出金額'] = df_concat['支出金額'].replace({',': '', '¥': ''}, regex=True).astype(int)

# 変換後の型を確認
print(df_concat[['日付', '支出金額']].dtypes)

日付      datetime64[ns]
支出金額             int64
dtype: object


In [62]:
df_concat.head()

,日付,科目,摘要,支出金額,処理対象pdfファイル名
0,2026-02-05,福利厚生費,新潟県けんこう財団_健康診断代,3686,Image_20260317_0002.pdf
1,2026-02-05,旅費交通費,ホテル法華クラブ新潟長岡_宿泊費,10300,Image_20260317_0002.pdf
2,2026-02-01,接待交際費,酒屋いって_飲食代,22400,Image_20260317_0002.pdf
3,2026-03-20,接待交際費,味処なべ_飲食代,12000,Image_20260317_0002.pdf
4,2026-02-20,旅費交通費,つばめタクシー_タクシー代,1400,Image_20260317_0002.pdf


## 科目コードの追加

In [63]:
# 科目とコードの関係を辞書で定義

code_book = {
    "運賃": 511,
    "支払手数料": 512,
    "諸会費": 520,
    "接待交際費": 521,
    "旅行交通費": 522,
    "通信費": 523,
    "事務消耗品費": 524,
    "消耗品費": 525,
    "租税公課": 526,
    "修繕費": 529,
    "保険料": 531,
    "燃料費": 533,
    "寄付金": 535,
    "管理諸費": 536,
    "図書研修費": 537,
    "雑費": 538,
    "医薬品費": 401,
    "福利厚生費": 430,
    "衛生管理費": 444
}

In [64]:
# '科目'列のインデックスを取得して、その隣（+1）に挿入
col_idx = df_concat.columns.get_loc('科目') + 1
df_concat.insert(col_idx, 'コード', df_concat['科目'].map(code_book))
display(df_concat.head())

,日付,科目,コード,摘要,支出金額,処理対象pdfファイル名
0,2026-02-05,福利厚生費,430.0,新潟県けんこう財団_健康診断代,3686,Image_20260317_0002.pdf
1,2026-02-05,旅費交通費,NaN,ホテル法華クラブ新潟長岡_宿泊費,10300,Image_20260317_0002.pdf
2,2026-02-01,接待交際費,521.0,酒屋いって_飲食代,22400,Image_20260317_0002.pdf
3,2026-03-20,接待交際費,521.0,味処なべ_飲食代,12000,Image_20260317_0002.pdf
4,2026-02-20,旅費交通費,NaN,つばめタクシー_タクシー代,1400,Image_20260317_0002.pdf


## 日付の付け替え

In [65]:
# '科目'列の値を元に'コード'列を作成
df_concat['備考'] = df_concat['日付']
display(df_concat.head())

,日付,科目,コード,摘要,支出金額,処理対象pdfファイル名,備考
0,2026-02-05,福利厚生費,430.0,新潟県けんこう財団_健康診断代,3686,Image_20260317_0002.pdf,2026-02-05
1,2026-02-05,旅費交通費,NaN,ホテル法華クラブ新潟長岡_宿泊費,10300,Image_20260317_0002.pdf,2026-02-05
2,2026-02-01,接待交際費,521.0,酒屋いって_飲食代,22400,Image_20260317_0002.pdf,2026-02-01
3,2026-03-20,接待交際費,521.0,味処なべ_飲食代,12000,Image_20260317_0002.pdf,2026-03-20
4,2026-02-20,旅費交通費,NaN,つばめタクシー_タクシー代,1400,Image_20260317_0002.pdf,2026-02-20


In [66]:
# 日付が2026-02-01以前であれば、日付を2026-02-01に変更する。
# そうでなければ、備考を空にする。

# 1. 2026-02-01以前（含む）の場合、日付を2026-02-01に変更
df_concat.loc[df_concat['日付'] <= '2026-02-01', '日付'] = pd.Timestamp('2026-02-01')

# 2. 2026-02-01より後（そうでなければ）の場合、備考を空にする
df_concat.loc[df_concat['日付'] > '2026-02-01', '備考'] = ""

display(df_concat)

,日付,科目,コード,摘要,支出金額,処理対象pdfファイル名,備考
0,2026-02-05,福利厚生費,430.0,新潟県けんこう財団_健康診断代,3686,Image_20260317_0002.pdf,NaT
1,2026-02-05,旅費交通費,NaN,ホテル法華クラブ新潟長岡_宿泊費,10300,Image_20260317_0002.pdf,NaT
2,2026-02-01,接待交際費,521.0,酒屋いって_飲食代,22400,Image_20260317_0002.pdf,2026-02-01
3,2026-03-20,接待交際費,521.0,味処なべ_飲食代,12000,Image_20260317_0002.pdf,NaT
4,2026-02-20,旅費交通費,NaN,つばめタクシー_タクシー代,1400,Image_20260317_0002.pdf,NaT
5,2026-02-17,旅費交通費,NaN,中越交通_タクシー代,1200,Image_20260317_0002.pdf,NaT
6,2026-02-22,旅費交通費,NaN,長岡タクシー_タクシー代,2400,Image_20260317_0002.pdf,NaT
7,2026-02-15,管理費,NaN,長岡市シルバー人材センター_駐車場整理,17092,Image_20260317_0001.pdf,NaT
8,2026-02-01,旅費交通費,NaN,タクシー代,1300,Image_20260317_0001.pdf,2025-02-27
9,2026-02-01,旅費交通費,NaN,長岡タクシー_タクシー代,1300,Image_20260317_0001.pdf,2026-01-24


## 編集したdf_concatをcsvファイルとして保存する


In [67]:
#####  編集したdf_concatをcsvファイルとして保存する

# 保存先のディレクトリパス
output_dir = '/content/drive/MyDrive/receipts/処理結果/2026-02/結合済みcsv'
file_name = '2026_02.csv'

# 保存先のフルパスを作成
save_path = os.path.join(output_dir, file_name)

# CSVとして保存
# index=False にすることで、DataFrameのインデックス（左端の数字列）を除外して保存できます
df_concat.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"保存が完了しました: {save_path}")

保存が完了しました: /content/drive/MyDrive/receipts/処理結果/2026-02/結合済みcsv/2026_02.csv
